# SalesLT - Phase validation

This read-only notebook produces evidence that Lakehouse Federation replication and the downstream Medallion layers agree. It compares source and Bronze snapshots, inspects Silver and Gold results, and reconciles financial measures without modifying pipeline tables.

## Validation configuration

The environment widget resolves the federated source catalog and all lakehouse targets. Successful source queries also provide practical evidence that the externally configured service-principal authentication and NCC/private connectivity path are operational.

In [0]:
dbutils.widgets.text(
    "environment",
    "dev",
    "Environment"
)

environment = dbutils.widgets.get("environment").lower()

if environment not in ["dev", "prod"]:
    raise ValueError(
        "Environment must be either 'dev' or 'prod'."
    )

config = {
    "dev": {
        "source_catalog": "fc_saleslt_dev",
        "catalog": "saleslt_dev"
    },
    "prod": {
        "source_catalog": "fc_saleslt_prod",
        "catalog": "saleslt_prod"
    }
}

env = config[environment]

source_catalog = env["source_catalog"]
catalog = env["catalog"]

# Bronze
customer_bronze = f"{catalog}.bronze.customer_raw"
product_bronze = f"{catalog}.bronze.product_raw"
category_bronze = f"{catalog}.bronze.product_category_raw"
header_bronze = f"{catalog}.bronze.sales_order_header_raw"
detail_bronze = f"{catalog}.bronze.sales_order_detail_raw"

# Silver
customer_silver = f"{catalog}.silver.customers"
product_silver = f"{catalog}.silver.products"
sales_silver = f"{catalog}.silver.sales_order_lines"

# Gold
product_gold = f"{catalog}.gold.sales_by_product"
customer_gold = f"{catalog}.gold.sales_by_customer"
monthly_gold = f"{catalog}.gold.monthly_sales_summary"

print("=" * 60)
print("SALESLT - PHASE VALIDATION")
print("=" * 60)
print(f"Environment    : {environment}")
print(f"Source catalog : {source_catalog}")
print(f"Target catalog : {catalog}")
print("=" * 60)

## Federation-to-Bronze snapshot reconciliation

Each Azure SQL source table is paired with its Bronze replica and counted. Equality confirms that the current federated snapshot was fully materialized; any mismatch is surfaced as validation evidence rather than hidden by downstream processing.

In [0]:
# COMMAND ----------

federation_bronze_mapping = {
    "Customer": (
        f"{source_catalog}.SalesLT.Customer",
        customer_bronze
    ),
    "Product": (
        f"{source_catalog}.SalesLT.Product",
        product_bronze
    ),
    "ProductCategory": (
        f"{source_catalog}.SalesLT.ProductCategory",
        category_bronze
    ),
    "SalesOrderHeader": (
        f"{source_catalog}.SalesLT.SalesOrderHeader",
        header_bronze
    ),
    "SalesOrderDetail": (
        f"{source_catalog}.SalesLT.SalesOrderDetail",
        detail_bronze
    )
}

validation_rows = []

for table_name, (
    source_table,
    bronze_table
) in federation_bronze_mapping.items():

    source_count = spark.table(
        source_table
    ).count()

    bronze_count = spark.table(
        bronze_table
    ).count()

    validation_rows.append(
        (
            table_name,
            source_count,
            bronze_count,
            (
                "PASS"
                if source_count == bronze_count
                else "FAIL"
            )
        )
    )

source_validation_df = spark.createDataFrame(
    validation_rows,
    [
        "table_name",
        "federation_count",
        "bronze_count",
        "status"
    ]
)

display(source_validation_df)

## Silver conformance evidence

Counts and representative order-line output demonstrate that replicated entities were standardized and joined into the published customer, product, and sales grains with normalized financial fields.

In [0]:

customer_count = spark.table(
    customer_silver
).count()

product_count = spark.table(
    product_silver
).count()

sales_count = spark.table(
    sales_silver
).count()

print("=" * 60)
print("SALESLT SILVER SUMMARY")
print("=" * 60)
print(f"Customers         : {customer_count}")
print(f"Products          : {product_count}")
print(f"Sales order lines : {sales_count}")
print("=" * 60)

In [0]:

display(
    spark.table(sales_silver)

        .select(
            "sales_order_detail_id",
            "sales_order_id",
            "customer_id",
            "customer_name",
            "product_id",
            "product_name",
            "product_category",
            "order_quantity",
            "unit_price",
            "net_line_amount"
        )

        .orderBy(
            "sales_order_detail_id"
        )

        .limit(30)
)

## Gold outputs and financial reconciliation

Ranked product and customer views plus monthly summaries provide reviewable business results. The final check compares total Silver line revenue with the corresponding Gold aggregate and fails explicitly if the layers do not reconcile.

In [0]:
display(
    spark.table(product_gold)
        .orderBy(
            "revenue_rank"
        )
        .limit(30)
)

display(
    spark.table(customer_gold)
        .orderBy(
            "total_spent",
            ascending=False
        )
        .limit(30)
)

display(
    spark.table(monthly_gold)
        .orderBy(
            "month_start_date"
        )
)

In [0]:


from pyspark.sql.functions import (
    sum,
    round
)

silver_revenue = (
    spark.table(sales_silver)
        .agg(
            round(
                sum("net_line_amount"),
                2
            ).alias("revenue")
        )
        .first()["revenue"]
)

product_revenue = (
    spark.table(product_gold)
        .agg(
            round(
                sum("net_revenue"),
                2
            ).alias("revenue")
        )
        .first()["revenue"]
)

monthly_revenue = (
    spark.table(monthly_gold)
        .agg(
            round(
                sum("net_revenue"),
                2
            ).alias("revenue")
        )
        .first()["revenue"]
)

print("=" * 60)
print("SALESLT GOLD RECONCILIATION")
print("=" * 60)
print(f"Silver revenue        : {silver_revenue}")
print(f"Product Gold revenue  : {product_revenue}")
print(f"Monthly Gold revenue  : {monthly_revenue}")
print()

if (
    silver_revenue
    == product_revenue
    == monthly_revenue
):
    print(
        "PASS - Gold revenue reconciliation successful."
    )
else:
    print(
        "FAIL - Gold revenue reconciliation mismatch."
    )

print("=" * 60)